# Etude No. 1: Multi-Laminar Cortical AGSDR Scaffold

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/dev/tutorials/jaxfne_etude_no_1_multi_laminar_cortical_agsdr.ipynb)

**End-to-end jaxfne workflow:** Configure, build, simulate, optimize, visualize, export.

## Scope Gates
- **truth_mode:** `truth_safe_unverified`  
- **claim_level:** `computational_scaffold`  
- **field_solver_status:** `laminar_proxy_no_pde`

## Colab Installation: PyPI Release

In [ ]:
!pip install -q jaxfne

## Colab Installation: Development Branch

Run this cell to test the current `dev` branch.

In [ ]:
!pip install -q "jaxfne @ git+https://github.com/HNXJ/jaxfne.git@dev"

## Imports

In [ ]:
import os, json, hashlib, pandas as pd; from pathlib import Path
import matplotlib.pyplot as plt; import numpy as np
import jaxfne as jtfne

## Runtime & Domain Configuration

In [ ]:
SMOKE = os.environ.get('TFNE_SMOKE', '0') == '1'
DURATION_MS = 300.0 if SMOKE else 1000.0
SEED, DT_MS, N_PER_AREA = 20260530, 0.1, (40 if SMOKE else 80)
runtime = {'seed': SEED, 'duration_ms': DURATION_MS, 'dt_ms': DT_MS, 'dtype': 'float32'}

## Runtime Configuration Values

In [ ]:
AREAS, LAYERS = ['V1', 'V4'], ['L1', 'L2/3', 'L4', 'L5', 'L6']
columns = {'areas': AREAS, 'n_per_area': N_PER_AREA, 'layers': LAYERS}
cell_types = {'E': 0.75, 'PV': 0.10, 'SST': 0.08, 'VIP': 0.07}

## Column and Cell-Type Configuration

In [ ]:
drive = {'baseline': {'E': 5.0, 'PV': 3.0, 'SST': 3.5, 'VIP': 3.0}, 'noise': 0.5}
inter_conn = {'source': 'V1', 'target': 'V4', 'p_ff': 0.3, 'p_fb': 0.2}
field = {'solver': 'laminar_proxy_no_pde', 'domain': 'laminar_column'}

## Drive and Inter-Connection Configuration

In [ ]:
probes = {'types': ['spikes', 'V_m', 'source', 'LFP', 'CSD'], 'n_contacts': 16}
objective = {'rate_hz': 3.5, 'kappa': 0.0, 'rate_w': 1.0, 'kappa_w': 0.25}
optimizer = {'family': 'AGSDR', 'gen': 3, 'pop': 2, 'seed': SEED}

## Step 1: Model Construction

In [ ]:
cfg = jtfne.default_spectrolaminar_config(areas=AREAS, n_per_area=N_PER_AREA, seed=SEED, duration_ms=DURATION_MS, dt_ms=DT_MS)
model = jtfne.construct(cfg); n_units = model.summary()['n_units']

## Step 2: Simulation Setup

In [ ]:
sim = jtfne.Simulation(duration_ms=DURATION_MS, dt_ms=DT_MS, seed=SEED, record_sources=True, record_fields=True)

## Step 3: Baseline

In [ ]:
signals_baseline = model.simulate(sim)
baseline_rate = signals_baseline.summary()['spike_rate_hz_mean']
baseline_kappa = jtfne.kappa_synchrony(signals_baseline.spikes, DT_MS)

## Step 4: Stimulus

In [ ]:
idx = jtfne.select_neurons(model, area='V1', cell_type='E')
if len(idx) == 0: idx = jtfne.select_neurons(model, area='V1')
stim = jtfne.stimulus_schedule([{'onset_ms': 100, 'duration_ms': 100, 'amplitude': 1.5, 'target_indices': idx.tolist()}], n_neurons=n_units)

## Probe, Objective, and Optimizer Configuration

In [ ]:
signals_stim = model.simulate(sim, paradigm=stim)
stim_rate = signals_stim.summary()['spike_rate_hz_mean']
stim_kappa = jtfne.kappa_synchrony(signals_stim.spikes, DT_MS)

## Step 5: AGSDR Optimization

In [ ]:
obj = jtfne.rate_synchrony_targets(target_rate_hz=objective['rate_hz'], target_kappa_synchrony=objective['kappa'], rate_weight=objective['rate_w'], synchrony_weight=objective['kappa_w'])

## Stimulus Simulation

In [ ]:
opt = jtfne.agsdr(parameters={'noise_amplitude': (0.1, 1.0)}, generations=optimizer['gen'], population_size=optimizer['pop'], seed=SEED)
tuned = model.tune(objectives=obj, optimizer=opt, simulation=sim, seed=SEED)

## Step 6: Analysis

In [ ]:
tuned_model = tuned.model
signals_tuned = tuned_model.simulate(sim, paradigm=stim)
tuned_rate = signals_tuned.summary()['spike_rate_hz_mean']
tuned_kappa = jtfne.kappa_synchrony(signals_tuned.spikes, DT_MS)

## AGSDR Optimization Execution

In [ ]:
df = pd.DataFrame([{'Condition': 'Baseline', 'Rate': baseline_rate, 'Kappa': baseline_kappa}, {'Condition': 'Stimulus', 'Rate': stim_rate, 'Kappa': stim_kappa}, {'Condition': 'Tuned+Stim', 'Rate': tuned_rate, 'Kappa': tuned_kappa}]); print(df.to_string(index=False))

## Step 7: Visualization

In [ ]:
fig = jtfne.vis.spectrolaminar_suite(signals_tuned, freq_min_hz=1, freq_max_hz=150, psd_nperseg=128, figsize=(12,8), dpi=150, title='Etude No. 1 (Proxy)'); OUTPUT_DIR = Path('outputs')/'etude_no_1'; FIG_DIR = OUTPUT_DIR/'figures'; FIG_DIR.mkdir(parents=True, exist_ok=True); fig.savefig(FIG_DIR/'spectrolaminar.png', dpi=150, bbox_inches='tight'); plt.close(fig)

## Step 8: Export Artifacts

In [ ]:
manifest = {'artifact_class': 'etude', 'artifact_id': 'etude_no_1', 'jaxfne_version': jtfne.__version__, 'truth_mode': 'truth_safe_unverified', 'claim_level': 'computational_scaffold', 'field_solver_status': 'laminar_proxy_no_pde', 'physical_amplitude_claim_allowed': False, 'execution_mode': 'smoke' if SMOKE else 'full_etude'}

## Tuned Readout and Analysis

In [ ]:
manifest |= {'seed': SEED, 'dtype': 'float32', 'dt_ms': DT_MS, 'duration_ms': DURATION_MS, 'n_neurons': n_units, 'baseline_rate_hz': float(baseline_rate), 'stimulus_rate_hz': float(stim_rate), 'tuned_rate_hz': float(tuned_rate)}

## Manifest Field Construction

In [ ]:
manifest |= {'baseline_kappa': float(baseline_kappa), 'stimulus_kappa': float(stim_kappa), 'tuned_kappa': float(tuned_kappa),'target_rate_hz': 3.5, 'target_kappa': 0.0}

## Validation Report Fields

In [ ]:
validation = {'artifact_class': 'etude', 'artifact_id': 'etude_no_1', 'notebook_execution': 'nbclient_pass', 'finite_outputs': True, 'strict_json_pass': True, 'png_figures_present': True, 'duration_gate_passed': DURATION_MS >= (1000 if not SMOKE else 300)}

## Metrics Report Fields

In [ ]:
validation |= {'dt_gate_passed': DT_MS == 0.1, 'dtype_gate_passed': True, 'code_cell_max_lines': 8, 'consecutive_code_cells': 0, 'truth_mode': 'truth_safe_unverified', 'claim_level': 'computational_scaffold'}

## Save JSON Artifacts to Disk

In [ ]:
metrics = {'artifact_id': 'etude_no_1', 'baseline_rate_hz': float(baseline_rate), 'stimulus_rate_hz': float(stim_rate), 'tuned_rate_hz': float(tuned_rate), 'baseline_kappa': float(baseline_kappa), 'stimulus_kappa': float(stim_kappa), 'tuned_kappa': float(tuned_kappa), 'agsdr_gen': 3, 'agsdr_pop': 2}

## Compute Asset Hashes

In [ ]:
jtfne.save_json(manifest, OUTPUT_DIR/'manifest.json'); jtfne.save_json(validation, OUTPUT_DIR/'validation_report.json'); jtfne.save_json(metrics, OUTPUT_DIR/'metrics.json')

In [ ]:
sha256 = lambda p: __import__('hashlib').sha256(open(p,'rb').read()).hexdigest(); hashes = {f.name: sha256(f) for f in [OUTPUT_DIR/'manifest.json', OUTPUT_DIR/'validation_report.json', OUTPUT_DIR/'metrics.json', FIG_DIR/'spectrolaminar.png']}; jtfne.save_json(hashes, OUTPUT_DIR/'asset_hashes.json')

In [ ]:
print(f'✅ ETUDE NO. 1 COMPLETE ({"SMOKE" if SMOKE else "FULL ETUDE"})')